# Sparsity Experiment — Model Degradation and Crossover Analysis

How does recommendation quality degrade as the dataset gets sparser, and at what point does content-based beat penalized SVD?

Six sparsity levels: 1.0, 0.5, 0.1, 0.01, 0.001, 0.0001. Models: popularity baseline, SVD, penalized SVD (λ=0.3), content-based. Metrics: RMSE, NDCG@10, catalog coverage, long-tail coverage with 95% bootstrap CIs.

> Run `04_content_based.ipynb` first.

In [ ]:
# imports
import numpy as np
import pandas as pd
import random
import warnings
import os
import joblib

import matplotlib
matplotlib.use('Agg')          # headless backend — safe for both notebook & script
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from scipy import stats
from tqdm.notebook import tqdm

from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split as surprise_split

DATA_DIR   = '../../ml-25m/'
OUTPUT_DIR = '../outputs/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_SEED    = 42
N_EVAL_USERS   = 500    # users sampled for NDCG / coverage evaluation
N_BOOTSTRAP    = 1_000  # bootstrap resamples for confidence intervals
TOP_N          = 10     # recommendation list length
LAMBDA_PENALTY = 0.3    # popularity penalty coefficient (from 02_models.ipynb)

In [ ]:
# load data and pre-trained content-based model
# we load the full 25M rating file once here.  At each sparsity level we
# subsample in-memory via DataFrame.sample(), avoiding repeated disk I/O.

print('Loading full ratings dataset (25M rows)...')
full_ratings = pd.read_csv(DATA_DIR + 'ratings.csv')
print(f'  Loaded {len(full_ratings):,} ratings')

# load the TF-IDF model built in 04_content_based.ipynb.
# this model is fixed — it does not depend on rating data — so it serves
# as a constant benchmark across all sparsity levels.
print('Loading content-based model...')
cb_artefact   = joblib.load(OUTPUT_DIR + 'content_based_model.pkl')
tfidf_matrix  = cb_artefact['tfidf_matrix']    # sparse, L2-normalised
movie_ids_cb  = cb_artefact['movie_ids']        # numpy array
mid_to_idx_cb = cb_artefact['movie_id_to_idx']  # dict
print(f'  TF-IDF matrix: {tfidf_matrix.shape}')

In [ ]:
def ndcg_at_k(recommended, relevant_ratings, k=10):
    """
    Graded NDCG@k. Relevance = rating/5.0, so a 5-star film is 10x more
    relevant than a 0.5-star film. Returns 0 if user has no relevant test items.
    """
    if not relevant_ratings:
        return 0.0

    dcg = sum(
        (relevant_ratings[item] / 5.0) / np.log2(rank + 2)
        for rank, item in enumerate(recommended[:k])
        if item in relevant_ratings
    )

    ideal_rels = sorted(relevant_ratings.values(), reverse=True)[:k]
    idcg = sum(
        (rel / 5.0) / np.log2(rank + 2)
        for rank, rel in enumerate(ideal_rels)
    )

    return dcg / idcg if idcg > 0 else 0.0


def catalog_coverage(recs_dict, all_movie_ids):
    """% of all films appearing in at least one top-N list."""
    if not all_movie_ids:
        return 0.0
    recommended = {mid for lst in recs_dict.values() for mid in lst}
    return len(recommended) / len(all_movie_ids) * 100


def longtail_coverage(recs_dict, tail_ids):
    """% of tail films appearing in at least one top-N list."""
    if not tail_ids:
        return 0.0
    recommended = {mid for lst in recs_dict.values() for mid in lst}
    return len(recommended & tail_ids) / len(tail_ids) * 100


def bootstrap_ci(values, n_boot=N_BOOTSTRAP, ci=0.95):
    """Non-parametric bootstrap CI for the mean. Fixed RNG seed for reproducibility."""
    if len(values) == 0:
        return (np.nan, np.nan)
    rng  = np.random.default_rng(RANDOM_SEED)
    arr  = np.asarray(values, dtype=float)
    boot = [np.mean(rng.choice(arr, size=len(arr), replace=True))
            for _ in range(n_boot)]
    alpha = (1 - ci) / 2
    return (float(np.percentile(boot, alpha * 100)),
            float(np.percentile(boot, (1 - alpha) * 100)))


print('Helper functions defined.')

In [ ]:
# recommendation functions

def svd_recs(user_id, svd_model, candidate_ids, seen_ids, n=TOP_N):
    """
    Standard SVD: rank candidates by predicted rating descending.
    Candidates are all movies present in the training set that the user
    has not already rated.
    """
    unrated = [mid for mid in candidate_ids if mid not in seen_ids]
    preds   = [svd_model.predict(user_id, mid) for mid in unrated]
    preds.sort(key=lambda p: p.est, reverse=True)
    return [p.iid for p in preds[:n]]


def penalized_svd_recs(user_id, svd_model, candidate_ids, seen_ids,
                       pop_dict, lam=LAMBDA_PENALTY, n=TOP_N):
    """
    Penalised SVD: adjust predicted score by popularity penalty before ranking.

    adjusted_score = cf_score * (1 - λ * pop_score)

    where pop_score ∈ [0,1] is the log-normalised rating count from the
    current training subset.  Films absent from pop_dict receive pop_score=0
    (no penalty), which is correct for new/unseen items.
    """
    unrated  = [mid for mid in candidate_ids if mid not in seen_ids]
    preds    = [svd_model.predict(user_id, mid) for mid in unrated]
    adjusted = [
        (p.iid, p.est * (1 - lam * pop_dict.get(p.iid, 0.0)))
        for p in preds
    ]
    adjusted.sort(key=lambda x: x[1], reverse=True)
    return [iid for iid, _ in adjusted[:n]]


def content_based_recs(user_id, train_df, tfidf_mat, mids, mid_to_idx,
                       n=TOP_N):
    """
    Content-based: weighted TF-IDF profile → cosine similarity ranking.
    Identical to the function defined in 04_content_based.ipynb.

    Note: the TF-IDF model itself does NOT change with sparsity level.  Only
    the user's seed ratings (drawn from the current training subset) change,
    which is the correct behaviour — fewer seeds = less personalised profile.
    """
    user_df = train_df[train_df['userId'] == user_id]
    if user_df.empty:
        return []
    seen_ids = set(user_df['movieId'].astype(int))
    profile  = np.zeros(tfidf_mat.shape[1], dtype=np.float32)
    for _, row in user_df.iterrows():
        mid = int(row['movieId'])
        if mid in mid_to_idx:
            profile += (float(row['rating']) / 5.0) * \
                       tfidf_mat[mid_to_idx[mid]].toarray().ravel()
    norm = np.linalg.norm(profile)
    if norm == 0:
        return []
    profile /= norm
    sims   = tfidf_mat.dot(profile)
    ranked = sorted(
        [(int(mids[i]), float(sims[i]))
         for i in range(len(mids)) if int(mids[i]) not in seen_ids],
        key=lambda x: x[1], reverse=True
    )
    return [mid for mid, _ in ranked[:n]]


print('Recommendation functions defined.')

---
## Experiment A: Growing Platform (Random Rating Subsample)

Uniformly random fraction of all 25M ratings at each sparsity level — shrinks both the number of ratings and unique movies. Content-based always uses the full 62,423-film catalogue; SVD is limited to movies in the subsample.

In [ ]:
# main loop

sparsity_levels = [1.0, 0.5, 0.1, 0.01, 0.001, 0.0001]
results_rows    = []
per_user_ndcg   = {}

FULL_CATALOGUE_SIZE = tfidf_matrix.shape[0]   # 62,423 — fixed for CB model
MODEL_NAMES = ['popularity_baseline', 'svd', 'penalized_svd', 'content_based']

for sparsity in tqdm(sparsity_levels, desc='Sparsity level'):

    n_ratings = max(int(len(full_ratings) * sparsity), 200)
    subset    = full_ratings.sample(n=n_ratings, random_state=RANDOM_SEED)

    n_unique_movies = subset['movieId'].nunique()
    n_unique_users  = subset['userId'].nunique()
    pct_catalogue   = n_unique_movies / FULL_CATALOGUE_SIZE * 100

    print(f'\n{"="*65}')
    print(f'SPARSITY = {sparsity}')
    print(f'{"="*65}')
    print(f'  Total ratings in subsample : {len(subset):>10,}')
    print(f'  Unique users               : {n_unique_users:>10,}')
    print(f'  Unique movies (subsample)  : {n_unique_movies:>10,}  '
          f'({pct_catalogue:.1f}% of full catalogue)')
    print(f'  Full catalogue size        : {FULL_CATALOGUE_SIZE:>10,}  '
          f'(TF-IDF matrix — ALWAYS covers all {FULL_CATALOGUE_SIZE:,} films)')
    print(f'  Movies NOT in subsample    : {FULL_CATALOGUE_SIZE - n_unique_movies:>10,}  '
          f'(CB can still recommend these; SVD cannot)')

    try:
        reader   = Reader(rating_scale=(0.5, 5.0))
        sur_data = Dataset.load_from_df(
            subset[['userId', 'movieId', 'rating']], reader
        )
        trainset, testset = surprise_split(
            sur_data, test_size=0.2, random_state=RANDOM_SEED
        )
    except Exception as exc:
        warnings.warn(f'  Data split failed at sparsity {sparsity}: {exc}')
        continue

    trainset_df = pd.DataFrame(
        [(int(trainset.to_raw_uid(u)),
          int(trainset.to_raw_iid(i)),
          float(r))
         for u, i, r in trainset.all_ratings()],
        columns=['userId', 'movieId', 'rating'],
    )
    testset_df = pd.DataFrame(
        [(int(uid), int(iid), float(r)) for uid, iid, r in testset],
        columns=['userId', 'movieId', 'rating'],
    )

    film_pop = (
        trainset_df.groupby('movieId')
        .agg(rating_count=('rating', 'count'))
        .reset_index()
        .sort_values('rating_count', ascending=False)
        .reset_index(drop=True)
    )
    log_c     = np.log1p(film_pop.set_index('movieId')['rating_count'])
    pop_dict  = (log_c / log_c.max()).to_dict() if len(log_c) > 0 else {}

    film_pop['cumpct'] = (film_pop['rating_count'].cumsum() /
                          film_pop['rating_count'].sum() * 100)
    if len(film_pop) > 1:
        cutoff  = (film_pop['cumpct'] >= 80).idxmax()
        tail_ids = set(film_pop.loc[cutoff + 1:, 'movieId'])
    else:
        tail_ids = set()

    all_train_mids  = set(trainset_df['movieId'].unique())
    pop_top10       = film_pop.head(TOP_N)['movieId'].tolist()

    svd_model = None
    rmse_svd  = np.nan
    try:
        svd_model = SVD(n_factors=100, random_state=RANDOM_SEED)
        svd_model.fit(trainset)
        svd_preds = svd_model.test(testset)
        rmse_svd  = accuracy.rmse(svd_preds, verbose=False)
        print(f'  SVD trained  RMSE={rmse_svd:.4f}')
    except Exception as exc:
        warnings.warn(f'  SVD training failed at sparsity {sparsity}: {exc}')

    train_users  = set(trainset_df['userId'].unique())
    test_users   = set(testset_df['userId'].unique())
    eval_users   = list(train_users & test_users)
    random.seed(RANDOM_SEED)
    eval_sample  = random.sample(eval_users, min(N_EVAL_USERS, len(eval_users)))
    print(f'  Evaluation users: {len(eval_sample)}')

    test_lookup  = (
        testset_df.groupby('userId')
        .apply(lambda x: dict(zip(x['movieId'], x['rating'])))
        .to_dict()
    )
    train_lookup = (
        trainset_df.groupby('userId')['movieId']
        .apply(set).to_dict()
    )

    user_ndcg     = {m: [] for m in MODEL_NAMES}
    user_recs_all = {m: {} for m in MODEL_NAMES}

    for uid in eval_sample:
        relevant = test_lookup.get(uid, {})
        if not relevant:
            continue
        seen = train_lookup.get(uid, set())

        pb = [m for m in pop_top10 if m not in seen]
        user_ndcg['popularity_baseline'].append(ndcg_at_k(pb, relevant))
        user_recs_all['popularity_baseline'][uid] = pb

        if svd_model is not None:
            try:
                sr = svd_recs(uid, svd_model, all_train_mids, seen)
                user_ndcg['svd'].append(ndcg_at_k(sr, relevant))
                user_recs_all['svd'][uid] = sr

                pr = penalized_svd_recs(uid, svd_model, all_train_mids,
                                        seen, pop_dict)
                user_ndcg['penalized_svd'].append(ndcg_at_k(pr, relevant))
                user_recs_all['penalized_svd'][uid] = pr
            except Exception as exc:
                warnings.warn(f'  SVD rec failed uid={uid}: {exc}')

        user_train_df = trainset_df[trainset_df['userId'] == uid]
        cb = content_based_recs(
            uid, user_train_df, tfidf_matrix, movie_ids_cb, mid_to_idx_cb
        )
        user_ndcg['content_based'].append(ndcg_at_k(cb, relevant))
        user_recs_all['content_based'][uid] = cb

    for model in MODEL_NAMES:
        ndcg_vals = np.array(user_ndcg[model])
        recs_dict = user_recs_all[model]

        mean_ndcg          = float(np.mean(ndcg_vals)) if len(ndcg_vals) > 0 else np.nan
        ci_lo, ci_hi       = bootstrap_ci(ndcg_vals)
        cat_cov            = catalog_coverage(recs_dict, all_train_mids)
        lt_cov             = longtail_coverage(recs_dict, tail_ids)
        rmse_val = rmse_svd if model in ('svd', 'penalized_svd') else np.nan

        results_rows.append({
            'sparsity_level'     : sparsity,
            'model'              : model,
            'rmse'               : round(rmse_val, 4) if not np.isnan(rmse_val) else np.nan,
            'ndcg10'             : round(mean_ndcg, 4),
            'ndcg10_ci_lower'    : round(ci_lo, 4),
            'ndcg10_ci_upper'    : round(ci_hi, 4),
            'catalogue_coverage' : round(cat_cov, 4),
            'long_tail_coverage' : round(lt_cov, 4),
        })

        per_user_ndcg[(sparsity, model)] = ndcg_vals.tolist()

        print(f'  [{model:<22}]  ndcg10={mean_ndcg:.4f}  '
              f'cat_cov={cat_cov:.2f}%  lt_cov={lt_cov:.2f}%')

print('\n✓ Experiment complete.')

In [ ]:
# save results to CSV

results_df = pd.DataFrame(results_rows)
out_csv    = OUTPUT_DIR + 'sparsity_expA_all_models.csv'
results_df.to_csv(out_csv, index=False)
print(f'Saved  →  {out_csv}')
print(f'Shape  :  {results_df.shape}')
print()
print(results_df.to_string(index=False))

In [ ]:
# preliminary NDCG@10 plot (without crossover line — redrawn in sp-c-11 once
# the crossover threshold is computed).

fig, ax = plt.subplots(figsize=(10, 5))

for model in MODEL_NAMES:
    sub = results_df[results_df['model'] == model].sort_values('sparsity_level')
    if sub.empty:
        continue

    x    = sub['sparsity_level'].values
    y    = sub['ndcg10'].values
    y_lo = sub['ndcg10_ci_lower'].values
    y_hi = sub['ndcg10_ci_upper'].values

    ax.plot(x, y,
            color=COLOURS[model], marker=MARKERS[model],
            label=LABELS[model], linewidth=2, markersize=7)
    ax.fill_between(x, y_lo, y_hi,
                    color=COLOURS[model], alpha=0.15)

ax.set_xscale('log')
ax.set_xlabel('Sparsity Level (fraction of full dataset)', fontsize=12)
ax.set_ylabel('NDCG@10', fontsize=12)
ax.set_title('NDCG@10 vs. Dataset Sparsity\n'
             '(shaded = 95% bootstrap CI, n=500 users)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# figure 2: Catalogue Coverage curves

fig, ax = plt.subplots(figsize=(10, 5))

for model in MODEL_NAMES:
    sub = results_df[results_df['model'] == model].sort_values('sparsity_level')
    if sub.empty:
        continue
    ax.plot(
        sub['sparsity_level'], sub['catalogue_coverage'],
        color=COLOURS[model], marker=MARKERS[model],
        label=LABELS[model], linewidth=2, markersize=7,
    )

if 'crossover_sparsity' in dir():
    ax.axvline(crossover_sparsity, color='black', linestyle='--',
               linewidth=1.2, label=f'Crossover @ {crossover_sparsity}')

ax.set_xscale('log')
ax.set_xlabel('Sparsity Level (fraction of full dataset)', fontsize=12)
ax.set_ylabel('Catalogue Coverage (%)', fontsize=12)
ax.set_title('Catalogue Coverage vs. Dataset Sparsity', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()

cov_fig_path = OUTPUT_DIR + 'sparsity_expA_coverage_curves.png'
plt.savefig(cov_fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved  →  {cov_fig_path}')

In [ ]:
# crossover analysis
#
# we define the crossover point as the highest (least sparse) sparsity level
# at which content-based NDCG@10 >= penalised SVD NDCG@10, scanning from the
# sparsest level upward.  This identifies the density threshold below which
# interaction-based models degrade past the content-based fallback.

pivot = results_df.pivot(index='sparsity_level', columns='model',
                         values='ndcg10').reset_index()
pivot = pivot.sort_values('sparsity_level')   # ascending: most sparse first

crossover_sparsity = None
crossover_row      = None

for _, row in pivot.iterrows():
    cb_val  = row.get('content_based',  np.nan)
    pen_val = row.get('penalized_svd',  np.nan)
    if pd.notna(cb_val) and pd.notna(pen_val) and cb_val > pen_val:
        crossover_sparsity = row['sparsity_level']
        crossover_row      = row
    else:
        # once penalised SVD takes the lead again (as sparsity increases),
        # stop — we want the last crossover as we move from sparse → dense.
        if crossover_sparsity is not None:
            break

if crossover_sparsity is not None:
    n_crossover = int(len(full_ratings) * crossover_sparsity)
    pct_density = crossover_sparsity * 100
    print('=' * 70)
    print('CROSSOVER THRESHOLD: SVD becomes less suitable than content-based')
    print(f'filtering below {pct_density:.4f}% of original rating density '
          f'({n_crossover:,} ratings)')
    print('=' * 70)
    print(f'  Content-based NDCG@10  : {crossover_row["content_based"]:.4f}')
    print(f'  Penalised SVD  NDCG@10 : {crossover_row["penalized_svd"]:.4f}')
else:
    print('No crossover found within the evaluated sparsity range.')
    print('Content-based never exceeded penalised SVD across these levels.')

In [ ]:
# mann-Whitney U test at the crossover sparsity level
#
# we compare the per-user NDCG@10 distributions of content-based and
# penalised SVD at the crossover sparsity level.
#
# H0: the two distributions are stochastically equal.
# H1 (alternative='greater'): content-based NDCG@10 is stochastically greater
#    than penalised SVD NDCG@10 at this sparsity level.
#
# mann-Whitney U is used rather than a paired t-test because per-user NDCG
# scores are bounded in [0,1] and right-skewed, violating normality.

if crossover_sparsity is not None:
    cb_scores  = per_user_ndcg.get((crossover_sparsity, 'content_based'),  [])
    pen_scores = per_user_ndcg.get((crossover_sparsity, 'penalized_svd'),  [])

    if len(cb_scores) > 0 and len(pen_scores) > 0:
        u_stat, p_value = stats.mannwhitneyu(
            cb_scores, pen_scores, alternative='greater'
        )
        significant = p_value < 0.05

        print(f'Mann-Whitney U test at crossover sparsity {crossover_sparsity}')
        print(f'  U statistic             : {u_stat:,.0f}')
        print(f'  P-value                 : {p_value:.4e}')
        print(f'  Significant (α=0.05)    : {significant}')
        print(f'  Content-based mean NDCG : {np.mean(cb_scores):.4f}')
        print(f'  Penalised SVD mean NDCG : {np.mean(pen_scores):.4f}')

        # save to CSV
        mw_out = OUTPUT_DIR + 'sparsity_expA_crossover_mannwhitney.csv'
        pd.DataFrame([{
            'crossover_sparsity_level'   : crossover_sparsity,
            'n_ratings_at_crossover'     : int(len(full_ratings) * crossover_sparsity),
            'u_statistic'                : u_stat,
            'p_value'                    : p_value,
            'significant_at_0.05'        : significant,
            'content_based_mean_ndcg10'  : np.mean(cb_scores),
            'penalized_svd_mean_ndcg10'  : np.mean(pen_scores),
            'n_users_content_based'      : len(cb_scores),
            'n_users_penalized_svd'      : len(pen_scores),
        }]).to_csv(mw_out, index=False)
        print(f'\nSaved  →  {mw_out}')
    else:
        print('Insufficient per-user scores for Mann-Whitney test '
              '(one or both models produced no valid NDCG scores at this level).')
else:
    print('No crossover identified — Mann-Whitney test skipped.')

In [ ]:
# redraw Figure 1 with crossover line

fig, ax = plt.subplots(figsize=(10, 5))

for model in MODEL_NAMES:
    sub = results_df[results_df['model'] == model].sort_values('sparsity_level')
    if sub.empty:
        continue
    x    = sub['sparsity_level'].values
    y    = sub['ndcg10'].values
    y_lo = sub['ndcg10_ci_lower'].values
    y_hi = sub['ndcg10_ci_upper'].values
    ax.plot(x, y, color=COLOURS[model], marker=MARKERS[model],
            label=LABELS[model], linewidth=2, markersize=7)
    ax.fill_between(x, y_lo, y_hi, color=COLOURS[model], alpha=0.15)

if crossover_sparsity is not None:
    ax.axvline(crossover_sparsity, color='black', linestyle='--',
               linewidth=1.5,
               label=f'Crossover threshold ({crossover_sparsity:.4f})')

ax.set_xscale('log')
ax.set_xlabel('Sparsity Level (fraction of full dataset)', fontsize=12)
ax.set_ylabel('NDCG@10', fontsize=12)
ax.set_title('NDCG@10 vs. Dataset Sparsity\n'
             '(shaded = 95% bootstrap CI, n=500 users)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'sparsity_expA_ndcg_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 (final with crossover line) saved.')

---
## Experiment B: Fixed Catalogue

Instead of random sampling, we keep a fixed fraction of each movie's ratings (min 1 per movie). All 62,423 films are present at every sparsity level — only the density of collaborative signal changes. This removes the confound in Experiment A where SVD loses access to many films at low sparsity.

In [ ]:
# experiment B: setup

# fixed tail IDs derived from full dataset (constant across all levels)
print('Computing fixed tail IDs from full dataset...')
_fp_full = (
    full_ratings.groupby('movieId')
    .agg(rating_count=('rating', 'count'))
    .reset_index()
    .sort_values('rating_count', ascending=False)
    .reset_index(drop=True)
)
_fp_full['cumpct'] = (_fp_full['rating_count'].cumsum() /
                      _fp_full['rating_count'].sum() * 100)
_cutoff_full    = (_fp_full['cumpct'] >= 80).idxmax()
tail_ids_fixed  = set(_fp_full.loc[_cutoff_full + 1:, 'movieId'])
head_ids_fixed  = set(_fp_full.loc[:_cutoff_full, 'movieId'])
print(f'  Full catalogue : {FULL_CATALOGUE_SIZE:,} films')
print(f'  Head (top 80%) : {len(head_ids_fixed):,} films')
print(f'  Tail (bottom 20%): {len(tail_ids_fixed):,} films')


# per-movie subsampling
def fixed_catalogue_sample(df, sparsity, random_seed=RANDOM_SEED):
    """
    Sample a fraction of each movie's ratings, keeping at least 1 per movie.

    This preserves all 62,423 films in the subsample regardless of sparsity,
    so both SVD and the content-based model operate over the same catalogue.
    The only thing varying across sparsity levels is how many ratings each
    movie has to estimate collaborative factors from.
    """
    return (
        df.groupby('movieId', group_keys=False)
        .apply(lambda x: x.sample(
            n=max(1, int(len(x) * sparsity)),
            random_state=random_seed
        ))
        .reset_index(drop=True)
    )


# batch SVD prediction helpers
# with ~62K candidate films per user, individual svd.predict() calls would be
# prohibitively slow (~hours).  Instead we use one matrix-vector multiply per
# user:  predicted_rating = global_mean + bu[u] + bi + qi @ pu[u]
# this is O(n_items × n_factors) per user vs O(n_items) Python calls.

def build_inner_to_raw(trainset):
    """Return array mapping inner item index → raw movieId (int)."""
    return np.array([int(trainset.to_raw_iid(i))
                     for i in range(trainset.n_items)], dtype=np.int64)


def svd_recs_batch(user_id, svd_model, inner_to_raw_arr, seen_ids, n=TOP_N):
    """
    Batch SVD top-N for one user.

    Parameters
    ----------
    user_id        : raw user ID (int)
    svd_model      : trained Surprise SVD object
    inner_to_raw_arr: numpy int64 array, inner item idx → raw movieId
    seen_ids       : set of movie IDs already rated by this user
    """
    ts = svd_model.trainset
    try:
        inner_uid = ts.to_inner_uid(str(user_id))
    except ValueError:
        return []   # user unseen in training

    # vectorised score computation
    scores = (ts.global_mean
              + svd_model.bu[inner_uid]
              + svd_model.bi
              + svd_model.qi.dot(svd_model.pu[inner_uid]))  # shape: (n_items,)

    # mask seen items and rank
    raw_ids = inner_to_raw_arr
    ranked  = sorted(
        [(int(raw_ids[i]), float(scores[i]))
         for i in range(ts.n_items) if int(raw_ids[i]) not in seen_ids],
        key=lambda x: x[1], reverse=True
    )
    return [mid for mid, _ in ranked[:n]]


def penalized_svd_recs_batch(user_id, svd_model, inner_to_raw_arr,
                              seen_ids, pop_arr, lam=LAMBDA_PENALTY, n=TOP_N):
    """
    Batch popularity-penalised SVD top-N for one user.

    Parameters
    ----------
    pop_arr : numpy float32 array of pop_scores, indexed by inner item idx.
              Precompute once per sparsity level:
              pop_arr = np.array([pop_dict.get(inner_to_raw[i], 0.) ...])
    """
    ts = svd_model.trainset
    try:
        inner_uid = ts.to_inner_uid(str(user_id))
    except ValueError:
        return []

    scores = (ts.global_mean
              + svd_model.bu[inner_uid]
              + svd_model.bi
              + svd_model.qi.dot(svd_model.pu[inner_uid]))
    adj_scores = scores * (1.0 - lam * pop_arr)  # vectorised penalty

    raw_ids = inner_to_raw_arr
    ranked  = sorted(
        [(int(raw_ids[i]), float(adj_scores[i]))
         for i in range(ts.n_items) if int(raw_ids[i]) not in seen_ids],
        key=lambda x: x[1], reverse=True
    )
    return [mid for mid, _ in ranked[:n]]


# coverage helpers with fixed-catalogue denominator
def catalog_coverage_B(recs_dict, full_size=FULL_CATALOGUE_SIZE):
    """% of full 62,423-film catalogue appearing in at least one rec list."""
    recommended = {mid for lst in recs_dict.values() for mid in lst}
    return len(recommended) / full_size * 100

def longtail_coverage_B(recs_dict, tail_ids):
    """% of fixed tail films appearing in at least one rec list."""
    if not tail_ids:
        return 0.0
    recommended = {mid for lst in recs_dict.values() for mid in lst}
    return len(recommended & tail_ids) / len(tail_ids) * 100


print('Experiment B helpers defined.')

In [ ]:
# experiment B: main loop

results_rows_B  = []
per_user_ndcg_B = {}

for sparsity in tqdm(sparsity_levels, desc='Exp B — sparsity level'):

    # 1. Fixed-catalogue subsample
    subset_B = fixed_catalogue_sample(full_ratings, sparsity)

    # diagnostics
    n_unique_movies_B = subset_B['movieId'].nunique()
    n_unique_users_B  = subset_B['userId'].nunique()
    avg_ratings_movie = len(subset_B) / n_unique_movies_B

    print(f'\n{"="*65}')
    print(f'EXPERIMENT B  |  SPARSITY = {sparsity}')
    print(f'{"="*65}')
    print(f'  Total ratings in subsample : {len(subset_B):>10,}')
    print(f'  Unique users               : {n_unique_users_B:>10,}')
    print(f'  Unique movies in subsample : {n_unique_movies_B:>10,}  '
          f'(target: all {FULL_CATALOGUE_SIZE:,})')
    print(f'  Avg ratings per movie      : {avg_ratings_movie:>10.2f}')
    print(f'  CB model scope             : {FULL_CATALOGUE_SIZE:>10,}  '
          f'(fixed — full TF-IDF catalogue)')
    if n_unique_movies_B < FULL_CATALOGUE_SIZE:
        missing = FULL_CATALOGUE_SIZE - n_unique_movies_B
        print(f'  ⚠ Movies missing from subsample: {missing:,}  '
              f'(these had 0 ratings in full_ratings — impossible)')

    # 2. Surprise split
    try:
        reader_B   = Reader(rating_scale=(0.5, 5.0))
        sur_data_B = Dataset.load_from_df(
            subset_B[['userId', 'movieId', 'rating']], reader_B
        )
        trainset_B, testset_B = surprise_split(
            sur_data_B, test_size=0.2, random_state=RANDOM_SEED
        )
    except Exception as exc:
        warnings.warn(f'  [Exp B] Data split failed at sparsity {sparsity}: {exc}')
        continue

    # 3. Extract DataFrames
    trainset_df_B = pd.DataFrame(
        [(int(trainset_B.to_raw_uid(u)),
          int(trainset_B.to_raw_iid(i)),
          float(r))
         for u, i, r in trainset_B.all_ratings()],
        columns=['userId', 'movieId', 'rating'],
    )
    testset_df_B = pd.DataFrame(
        [(int(uid), int(iid), float(r)) for uid, iid, r in testset_B],
        columns=['userId', 'movieId', 'rating'],
    )

    n_svd_train_movies = trainset_df_B['movieId'].nunique()
    print(f'  SVD training movies        : {n_svd_train_movies:>10,}  '
          f'({n_svd_train_movies/FULL_CATALOGUE_SIZE*100:.1f}% of full catalogue)')

    # 4. Popularity stats (from training subset)
    film_pop_B = (
        trainset_df_B.groupby('movieId')
        .agg(rating_count=('rating', 'count'))
        .reset_index()
        .sort_values('rating_count', ascending=False)
        .reset_index(drop=True)
    )
    log_c_B   = np.log1p(film_pop_B.set_index('movieId')['rating_count'])
    pop_dict_B = (log_c_B / log_c_B.max()).to_dict() if len(log_c_B) > 0 else {}

    all_train_mids_B = set(trainset_df_B['movieId'].unique())
    pop_top10_B      = film_pop_B.head(TOP_N)['movieId'].tolist()

    # 5. Train SVD
    svd_model_B = None
    rmse_svd_B  = np.nan
    inner_to_raw_B = None
    pop_arr_B      = None

    try:
        svd_model_B = SVD(n_factors=100, random_state=RANDOM_SEED)
        svd_model_B.fit(trainset_B)
        svd_preds_B = svd_model_B.test(testset_B)
        rmse_svd_B  = accuracy.rmse(svd_preds_B, verbose=False)
        print(f'  SVD trained  RMSE={rmse_svd_B:.4f}  '
              f'({trainset_B.n_items:,} items in SVD)')

        # precompute inner→raw mapping and popularity array for batch scoring
        inner_to_raw_B = build_inner_to_raw(trainset_B)
        pop_arr_B = np.array(
            [pop_dict_B.get(int(inner_to_raw_B[i]), 0.0)
             for i in range(trainset_B.n_items)],
            dtype=np.float32
        )
    except Exception as exc:
        warnings.warn(f'  [Exp B] SVD failed at sparsity {sparsity}: {exc}')

    # 6. Evaluation users
    train_users_B = set(trainset_df_B['userId'].unique())
    test_users_B  = set(testset_df_B['userId'].unique())
    eval_users_B  = list(train_users_B & test_users_B)
    random.seed(RANDOM_SEED)
    eval_sample_B = random.sample(eval_users_B,
                                  min(N_EVAL_USERS, len(eval_users_B)))
    print(f'  Evaluation users: {len(eval_sample_B)}')

    test_lookup_B  = (
        testset_df_B.groupby('userId')
        .apply(lambda x: dict(zip(x['movieId'], x['rating'])))
        .to_dict()
    )
    train_lookup_B = (
        trainset_df_B.groupby('userId')['movieId'].apply(set).to_dict()
    )

    # 7. Recommendations
    user_ndcg_B     = {m: [] for m in MODEL_NAMES}
    user_recs_all_B = {m: {} for m in MODEL_NAMES}

    for uid in eval_sample_B:
        relevant = test_lookup_B.get(uid, {})
        if not relevant:
            continue
        seen = train_lookup_B.get(uid, set())

        # popularity baseline
        pb = [m for m in pop_top10_B if m not in seen]
        user_ndcg_B['popularity_baseline'].append(ndcg_at_k(pb, relevant))
        user_recs_all_B['popularity_baseline'][uid] = pb

        # SVD models (batch)
        if svd_model_B is not None and inner_to_raw_B is not None:
            try:
                sr = svd_recs_batch(uid, svd_model_B, inner_to_raw_B, seen)
                user_ndcg_B['svd'].append(ndcg_at_k(sr, relevant))
                user_recs_all_B['svd'][uid] = sr

                pr = penalized_svd_recs_batch(uid, svd_model_B,
                                              inner_to_raw_B, seen, pop_arr_B)
                user_ndcg_B['penalized_svd'].append(ndcg_at_k(pr, relevant))
                user_recs_all_B['penalized_svd'][uid] = pr
            except Exception as exc:
                warnings.warn(f'  [Exp B] SVD rec failed uid={uid}: {exc}')

        # content-based (model fixed; only user profile changes with sparsity)
        user_train_df_B = trainset_df_B[trainset_df_B['userId'] == uid]
        cb = content_based_recs(
            uid, user_train_df_B, tfidf_matrix, movie_ids_cb, mid_to_idx_cb
        )
        user_ndcg_B['content_based'].append(ndcg_at_k(cb, relevant))
        user_recs_all_B['content_based'][uid] = cb

    # 8 & 9. Aggregate + bootstrap CIs
    for model in MODEL_NAMES:
        ndcg_vals  = np.array(user_ndcg_B[model])
        recs_dict  = user_recs_all_B[model]

        mean_ndcg    = float(np.mean(ndcg_vals)) if len(ndcg_vals) > 0 else np.nan
        ci_lo, ci_hi = bootstrap_ci(ndcg_vals)
        cat_cov      = catalog_coverage_B(recs_dict)     # denominator = 62,423
        lt_cov       = longtail_coverage_B(recs_dict, tail_ids_fixed)
        rmse_val     = rmse_svd_B if model in ('svd', 'penalized_svd') else np.nan

        results_rows_B.append({
            'sparsity_level'     : sparsity,
            'model'              : model,
            'avg_ratings_per_movie': round(avg_ratings_movie, 2),
            'rmse'               : round(rmse_val, 4) if not np.isnan(rmse_val) else np.nan,
            'ndcg10'             : round(mean_ndcg, 4),
            'ndcg10_ci_lower'    : round(ci_lo, 4),
            'ndcg10_ci_upper'    : round(ci_hi, 4),
            'catalogue_coverage' : round(cat_cov, 4),
            'long_tail_coverage' : round(lt_cov, 4),
        })
        per_user_ndcg_B[(sparsity, model)] = ndcg_vals.tolist()

        print(f'  [{model:<22}]  ndcg10={mean_ndcg:.4f}  '
              f'cat_cov={cat_cov:.2f}%  lt_cov={lt_cov:.2f}%')

print('\n✓ Experiment B complete.')

In [ ]:
# experiment B: Save results to CSV
results_df_B = pd.DataFrame(results_rows_B)
out_csv_B    = OUTPUT_DIR + 'sparsity_expB_no_hybrid.csv'
results_df_B.to_csv(out_csv_B, index=False)
print(f'Saved  →  {out_csv_B}')
print(f'Shape  :  {results_df_B.shape}')
print()
print(results_df_B.to_string(index=False))

In [ ]:
# experiment B: Figure 1 — NDCG@10 curves

fig, ax = plt.subplots(figsize=(10, 5))

for model in MODEL_NAMES:
    sub = results_df_B[results_df_B['model'] == model].sort_values('sparsity_level')
    if sub.empty:
        continue
    x, y     = sub['sparsity_level'].values, sub['ndcg10'].values
    y_lo, y_hi = sub['ndcg10_ci_lower'].values, sub['ndcg10_ci_upper'].values
    ax.plot(x, y, color=COLOURS[model], marker=MARKERS[model],
            label=LABELS[model], linewidth=2, markersize=7)
    ax.fill_between(x, y_lo, y_hi, color=COLOURS[model], alpha=0.15)

if 'crossover_sparsity_B' in dir() and crossover_sparsity_B is not None:
    ax.axvline(crossover_sparsity_B, color='black', linestyle='--',
               linewidth=1.5, label=f'Crossover @ {crossover_sparsity_B}')

ax.set_xscale('log')
ax.set_xlabel('Sparsity Level (fraction of ratings per movie kept)', fontsize=12)
ax.set_ylabel('NDCG@10', fontsize=12)
ax.set_title('Experiment B: NDCG@10 vs. Rating Density\n'
             '(Fixed 62,423-film catalogue; shaded = 95% bootstrap CI, n=500 users)',
             fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()

ndcg_fig_B = OUTPUT_DIR + 'sparsity_expB_ndcg_curves.png'
plt.savefig(ndcg_fig_B, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved  →  {ndcg_fig_B}')

In [ ]:
# experiment B: Figure 2 — Catalogue Coverage curves
fig, ax = plt.subplots(figsize=(10, 5))

for model in MODEL_NAMES:
    sub = results_df_B[results_df_B['model'] == model].sort_values('sparsity_level')
    if sub.empty:
        continue
    ax.plot(sub['sparsity_level'], sub['catalogue_coverage'],
            color=COLOURS[model], marker=MARKERS[model],
            label=LABELS[model], linewidth=2, markersize=7)

if 'crossover_sparsity_B' in dir() and crossover_sparsity_B is not None:
    ax.axvline(crossover_sparsity_B, color='black', linestyle='--',
               linewidth=1.5, label=f'Crossover @ {crossover_sparsity_B}')

ax.set_xscale('log')
ax.set_xlabel('Sparsity Level (fraction of ratings per movie kept)', fontsize=12)
ax.set_ylabel('Catalogue Coverage (%)', fontsize=12)
ax.set_title('Experiment B: Catalogue Coverage vs. Rating Density\n'
             '(denominator = full 62,423-film catalogue)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()

cov_fig_B = OUTPUT_DIR + 'sparsity_expB_coverage_curves.png'
plt.savefig(cov_fig_B, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved  →  {cov_fig_B}')

In [ ]:
# experiment B: Crossover analysis

pivot_B = (results_df_B
           .pivot(index='sparsity_level', columns='model', values='ndcg10')
           .reset_index()
           .sort_values('sparsity_level'))

crossover_sparsity_B = None
crossover_row_B      = None

for _, row in pivot_B.iterrows():
    cb_val  = row.get('content_based', np.nan)
    pen_val = row.get('penalized_svd', np.nan)
    if pd.notna(cb_val) and pd.notna(pen_val) and cb_val > pen_val:
        crossover_sparsity_B = row['sparsity_level']
        crossover_row_B      = row
    else:
        if crossover_sparsity_B is not None:
            break

if crossover_sparsity_B is not None:
    n_crossover_B = int(len(full_ratings) * crossover_sparsity_B)
    avg_at_crossover_B = results_df_B[
        results_df_B['sparsity_level'] == crossover_sparsity_B
    ]['avg_ratings_per_movie'].iloc[0]
    print('=' * 70)
    print('EXPERIMENT B — CROSSOVER THRESHOLD')
    print(f'  Sparsity level        : {crossover_sparsity_B}')
    print(f'  Approx. total ratings : {n_crossover_B:,}')
    print(f'  Avg ratings/movie     : {avg_at_crossover_B:.2f}')
    print(f'  Content-based NDCG@10 : {crossover_row_B["content_based"]:.4f}')
    print(f'  Penalised SVD  NDCG@10: {crossover_row_B["penalized_svd"]:.4f}')
    print('=' * 70)
else:
    print('No crossover found within the evaluated sparsity range.')

In [ ]:
# experiment B: Mann-Whitney U test at crossover

if crossover_sparsity_B is not None:
    cb_scores_B  = per_user_ndcg_B.get((crossover_sparsity_B, 'content_based'), [])
    pen_scores_B = per_user_ndcg_B.get((crossover_sparsity_B, 'penalized_svd'), [])

    if len(cb_scores_B) > 0 and len(pen_scores_B) > 0:
        u_stat_B, p_value_B = stats.mannwhitneyu(
            cb_scores_B, pen_scores_B, alternative='greater'
        )
        significant_B = p_value_B < 0.05

        print(f'Mann-Whitney U (Exp B, sparsity={crossover_sparsity_B}):')
        print(f'  U statistic             : {u_stat_B:,.0f}')
        print(f'  P-value                 : {p_value_B:.4e}')
        print(f'  Significant (α=0.05)    : {significant_B}')
        print(f'  Content-based mean NDCG : {np.mean(cb_scores_B):.4f}')
        print(f'  Penalised SVD mean NDCG : {np.mean(pen_scores_B):.4f}')

        mw_out_B = OUTPUT_DIR + 'sparsity_expB_crossover_mannwhitney.csv'
        pd.DataFrame([{
            'experiment'                : 'B_fixed_catalogue',
            'crossover_sparsity_level'  : crossover_sparsity_B,
            'approx_total_ratings'      : int(len(full_ratings) * crossover_sparsity_B),
            'avg_ratings_per_movie'     : avg_at_crossover_B,
            'u_statistic'               : u_stat_B,
            'p_value'                   : p_value_B,
            'significant_at_0.05'       : significant_B,
            'content_based_mean_ndcg10' : np.mean(cb_scores_B),
            'penalized_svd_mean_ndcg10' : np.mean(pen_scores_B),
            'n_users_content_based'     : len(cb_scores_B),
            'n_users_penalized_svd'     : len(pen_scores_B),
        }]).to_csv(mw_out_B, index=False)
        print(f'\nSaved  →  {mw_out_B}')
    else:
        print('Insufficient per-user scores for Mann-Whitney test.')
else:
    print('No crossover identified — Mann-Whitney test skipped.')

In [ ]:
# side-by-side comparison: Experiment A vs Experiment B

exp_a_path = OUTPUT_DIR + 'sparsity_expA_all_models.csv'
if os.path.exists(exp_a_path):
    results_df_A = pd.read_csv(exp_a_path)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

    for ax, (df, title, exp_label) in zip(
        axes,
        [
            (results_df_A, 'Experiment A\n(Shrinking Catalogue)', 'A'),
            (results_df_B, 'Experiment B\n(Fixed Catalogue: 62,423 films)', 'B'),
        ]
    ):
        for model in MODEL_NAMES:
            sub = df[df['model'] == model].sort_values('sparsity_level')
            if sub.empty:
                continue
            x, y = sub['sparsity_level'].values, sub['ndcg10'].values
            ax.plot(x, y, color=COLOURS[model], marker=MARKERS[model],
                    label=LABELS[model], linewidth=2, markersize=7)
            if 'ndcg10_ci_lower' in sub.columns:
                ax.fill_between(x, sub['ndcg10_ci_lower'].values,
                                sub['ndcg10_ci_upper'].values,
                                color=COLOURS[model], alpha=0.12)

        # crossover markers
        if exp_label == 'A':
            mw_a_path = OUTPUT_DIR + 'sparsity_expA_crossover_mannwhitney.csv'
            if os.path.exists(mw_a_path):
                cs_a = pd.read_csv(mw_a_path)['crossover_sparsity_level'].iloc[0]
                ax.axvline(cs_a, color='black', linestyle='--', linewidth=1.3,
                           label=f'Crossover @ {cs_a}')
        else:
            if 'crossover_sparsity_B' in dir() and crossover_sparsity_B is not None:
                ax.axvline(crossover_sparsity_B, color='black', linestyle='--',
                           linewidth=1.3, label=f'Crossover @ {crossover_sparsity_B}')

        ax.set_xscale('log')
        ax.set_xlabel('Sparsity Level', fontsize=11)
        ax.set_ylabel('NDCG@10', fontsize=11)
        ax.set_title(title, fontsize=12)
        ax.legend(fontsize=9)
        ax.grid(True, linestyle='--', alpha=0.4)

    fig.suptitle('NDCG@10 vs. Sparsity: Shrinking vs. Fixed Catalogue\n'
                 '(shaded = 95% bootstrap CI, n=500 users)', fontsize=13)
    plt.tight_layout()

    side_path = OUTPUT_DIR + 'sparsity_expAB_side_by_side.png'
    plt.savefig(side_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved  →  {side_path}')
else:
    print(f'Experiment A results not found ({exp_a_path}) — side-by-side skipped.')